# NB5 - Distance accuracy: ground-plane geometry (GeoCalib) vs Depth Pro
Compares both distance methods against **tape-measured floor distances** on your own photos and writes the tables and plots you show the prof.

This notebook does not run the detector or any engine. It only needs the saved outputs of the Depth Pro and GeoCalib notebooks (plus a small pose model, to find the people). Runtime: GPU or CPU.

## What this notebook expects (read before running)

**1. One folder on Drive, with three sub-folders**
```
MyDrive/cv5570_risk/accuracy/
   images/   your photos (jpg/png)
   depth/    <name>.npz   <- from the Depth Pro notebook (unzip depth_pro_outputs.zip, take its 'depth' folder)
   camera/   <name>.json  <- from the GeoCalib notebook (its out/camera folder)
```
Results are written to `accuracy/results/`.

**2. The same photos in all three places.** Depth Pro and GeoCalib must have been run on the *same image files* (same pixels, same orientation). The file name (without extension) is how the outputs are matched to the photo.
* Your earlier runs do not match: Depth Pro saved `image1 ... image5`, GeoCalib saved the original long names, and `image2` (854x800) was not in the GeoCalib set.
* Easiest fix: rename your test photos once (`acc01.jpg`, `acc02.jpg`, ...) and run both notebooks on those.
* Otherwise, fill `depth_stem` / `camera_stem` in the table below to point to the differently named outputs.

**3. A tape-measured distance per photo**, measured **along the floor** between two spots:
* Either the feet of two people standing in the photo (fully visible, head to feet, standing upright), or two marks on the floor.
* Photos at a range of distances (say 2 m to 10 m) make the comparison more convincing. 6-8 photos is a workable minimum.

**4. The table in the next-but-two cell (`ENTRIES`)**, one line per photo:

| field | meaning |
|---|---|
| `image` | file name in `images/` |
| `true_m` | tape-measured floor distance in metres |
| `a`, `b` | the two end points. A **number** = that person, counted left to right (the preview cell shows the numbers); the point used is the middle of their ankles. Or a pixel `(u, v)` of a floor spot. Default `a=0, b=1` |
| `depth_stem`, `camera_stem` | only if the output files have a different name from the image |
| `cam_height_m` | only if you know the camera height above the floor (otherwise it is estimated from the standing people, each assumed 1.7 m) |

Run the preview cell first and check that the numbers sit on the right people or floor spots before trusting any error figure.

In [ ]:
!pip -q install "ultralytics>=8.4.0"
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob, json
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

ACC_DIR   = '/content/drive/MyDrive/cv5570_risk/accuracy'
IMG_DIR, DEPTH_DIR, CAM_DIR = f'{ACC_DIR}/images', f'{ACC_DIR}/depth', f'{ACC_DIR}/camera'
RES_DIR   = f'{ACC_DIR}/results'
os.makedirs(RES_DIR, exist_ok=True)

WORKER_H          = 1.7    # assumed height of a standing person, metres (the "ruler" for camera height)
MAX_HEIGHT_SPREAD = 0.30   # geometry is rejected if the people disagree on camera height by more than this
TOPDOWN_G         = 0.85   # geometry is rejected if the camera looks (almost) straight down
DEPTH_WIN         = 2      # Depth Pro value = median over a (2*2+1)^2 pixel window around the point
AGREE_TOL         = 25.0   # the two methods "agree" if they differ by <= this % of their mean
POSE_IMGSZ        = 1280

## Fill in your photos and tape measurements here

In [ ]:
ENTRIES = [
    # dict(image='acc01.jpg', true_m=4.0),                                   # two people, default a=0, b=1 (left to right)
    # dict(image='acc02.jpg', true_m=7.5, a=0, b=2),                         # people number 0 and 2
    # dict(image='acc03.jpg', true_m=5.0, a=(812, 655), b=(1190, 702)),      # two floor marks, as pixel (u, v)
    # dict(image='acc04.jpg', true_m=6.0, depth_stem='image4', camera_stem='0020443_jpg.rf.xxxx'),   # differently named outputs
    # dict(image='acc05.jpg', true_m=3.0, cam_height_m=1.5),                 # camera height known
]
assert ENTRIES, 'Add at least one line to ENTRIES above (see the commented examples).'

## Helpers: pose, camera geometry, Depth Pro lookup

In [ ]:
pose_model = YOLO('yolov8m-pose.pt')       # LOAD MODELS

def pose_persons(img):
    """all people found by the pose model, sorted left -> right"""
    r = pose_model.predict(img, imgsz=POSE_IMGSZ, conf=0.25, verbose=False)[0]
    out = []
    if r.boxes is not None and len(r.boxes):
        kps = r.keypoints.data.cpu().numpy()
        for i, b in enumerate(r.boxes):
            out.append(dict(box=[float(v) for v in b.xyxy[0]], kp=kps[i]))
    out.sort(key=lambda p: (p['box'][0] + p['box'][2]) / 2)
    return out

def feet_px(p):
    """middle of the two ankles (keypoints 15, 16) if visible, else bottom-centre of the box"""
    kp = p['kp']
    if kp is not None:
        a = [kp[i][:2] for i in (15, 16) if kp[i][2] > 0.5]
        if a: return tuple(float(v) for v in np.mean(a, axis=0))
    return ((p['box'][0] + p['box'][2]) / 2, p['box'][3])

# --- camera geometry ---
# GeoCalib's 'g' (as saved by the GeoCalib notebook) is the UP direction in the camera frame (x right, y down, z forward):
# for an upright photo g is about (0, -1, 0). The geometry below needs the DOWN direction, so g is negated here.
def g_down(cam):
    g = -np.array(cam['g'], float)
    return g / np.linalg.norm(g)

def topdown(cam):
    return g_down(cam)[2] > TOPDOWN_G

def ray(cam, u, v):
    return np.array([(u - cam['cx']) / cam['fx'], (v - cam['cy']) / cam['fy'], 1.0])

def floor_point(u, v, cam, h):
    """3D point (metres, camera frame) where the pixel's ray meets the floor plane  g.X = h"""
    g = g_down(cam); d = ray(cam, u, v)
    den = g @ d
    return None if den <= 0.02 else d * h / den     # den <= 0: pixel is at/above the horizon

def estimate_height(persons, cam):
    """Camera height above the floor, using each upright full-body person as a 1.7 m ruler.
    For one person:  head = foot - 1.7*g_down,  foot = h*a,  head = s*d_head   ->  solve for (s, h)."""
    g = g_down(cam); hs = []
    for p in persons:
        kp = p['kp']
        if kp is None or not (kp[15][2] > 0.5 and kp[16][2] > 0.5): continue    # both ankles visible
        x1, y1, x2, y2 = p['box']
        if (y2 - y1) < 1.3 * (x2 - x1): continue                                 # must look upright / standing
        d_f = ray(cam, *feet_px(p)); d_t = ray(cam, (x1 + x2) / 2, y1)
        den = g @ d_f
        if den <= 0.05: continue
        a = d_f / den
        sol, *_ = np.linalg.lstsq(np.stack([d_t, -a], axis=1), -WORKER_H * g, rcond=None)
        s, h = sol
        if s > 0 and h > 0.3: hs.append(float(h))
    if not hs: return None, None, 0
    hs = np.array(hs); med = float(np.median(hs))
    spread = float(np.median(np.abs(hs - med)) / med) if len(hs) > 1 else None
    return med, spread, len(hs)

def geometry_distance(pa, pb, cam, persons, h_override=None):
    """-> (distance_m or None, info dict)"""
    if topdown(cam):
        return None, dict(note='geometry: camera looks almost straight down')
    if h_override:
        h, spread, n = float(h_override), None, 0
    else:
        h, spread, n = estimate_height(persons, cam)
        if h is None:
            return None, dict(note='geometry: no upright full-body person with both ankles visible -> no scale')
        if spread is not None and spread > MAX_HEIGHT_SPREAD:
            return None, dict(note=f'geometry: people disagree on camera height ({spread:.0%} spread)')
    A, B = floor_point(*pa, cam, h), floor_point(*pb, cam, h)
    if A is None or B is None:
        return None, dict(note='geometry: a point is at or above the horizon')
    return float(np.linalg.norm(A - B)), dict(h=h, spread=spread, n_ref=n)

# --- Depth Pro ---
def depth_xyz(u, v, depth, f):
    Hd, Wd = depth.shape
    ui, vi = int(np.clip(round(u), 0, Wd - 1)), int(np.clip(round(v), 0, Hd - 1))
    Z = float(np.median(depth[max(0, vi - DEPTH_WIN):vi + DEPTH_WIN + 1, max(0, ui - DEPTH_WIN):ui + DEPTH_WIN + 1]))
    return np.array([(ui - Wd / 2) * Z / f, (vi - Hd / 2) * Z / f, Z])

def depthpro_distance(pa, pb, depth, f):
    return float(np.linalg.norm(depth_xyz(*pa, depth, f) - depth_xyz(*pb, depth, f)))

def resolve_point(spec, persons):
    if isinstance(spec, (int, np.integer)):
        if spec >= len(persons): raise IndexError(f'person {spec} not found (pose model found {len(persons)})')
        return feet_px(persons[spec])
    return (float(spec[0]), float(spec[1]))

## Step 1 - Preview: check that every end point sits where you think it does
Numbers are the people (left to right). The coloured dots are the end points that will be measured.

In [ ]:
def preview(entries, cols=2):
    n = len(entries); rows_ = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(8 * cols, 6 * rows_), squeeze=False)
    for ax in axes.ravel(): ax.axis('off')
    for ax, e in zip(axes.ravel(), entries):
        img = cv2.imread(f"{IMG_DIR}/{e['image']}")
        if img is None:
            ax.set_title(f"{e['image']}: NOT FOUND in {IMG_DIR}", color='red'); continue
        persons = pose_persons(img)
        vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(vis)
        for i, p in enumerate(persons):
            x1, y1, x2, y2 = p['box']
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec='lime', lw=2))
            ax.text(x1, y1 - 4, str(i), color='lime', fontsize=14, weight='bold')
            fu, fv = feet_px(p); ax.plot(fu, fv, 'o', color='lime', ms=4)
        try:
            for tag, spec in (('A', e.get('a', 0)), ('B', e.get('b', 1))):
                u, v = resolve_point(spec, persons)
                ax.plot(u, v, 'o', color='red', ms=10, mfc='none', mew=2); ax.text(u + 8, v, tag, color='red', fontsize=14, weight='bold')
            ax.set_title(f"{e['image']}  |  tape: {e['true_m']} m  |  {len(persons)} people found")
        except IndexError as ex:
            ax.set_title(f"{e['image']}: {ex}", color='red')
    plt.tight_layout()
    plt.savefig(f'{RES_DIR}/preview.png', dpi=80)
    plt.show()

preview(ENTRIES)

## Step 2 - Measure with both methods

In [ ]:
rows = []
for e in ENTRIES:
    name = e['image']; stem = os.path.splitext(name)[0]
    row = dict(image=name, true_m=float(e['true_m']), geometry_m=np.nan, depthpro_m=np.nan,
               cam_height_m=np.nan, n_ref_people=0, height_spread=np.nan,
               f_geocalib=np.nan, f_depthpro=np.nan, pitch_deg=np.nan)
    notes = []
    img = cv2.imread(f'{IMG_DIR}/{name}')
    if img is None:
        row['notes'] = 'image not found'; rows.append(row); continue
    H, W = img.shape[:2]
    persons = pose_persons(img)
    try:
        pa, pb = resolve_point(e.get('a', 0), persons), resolve_point(e.get('b', 1), persons)
    except IndexError as ex:
        row['notes'] = str(ex); rows.append(row); continue

    # geometry (GeoCalib)
    cj = f"{CAM_DIR}/{e.get('camera_stem', stem)}.json"
    if not os.path.exists(cj):
        notes.append(f'no GeoCalib file {os.path.basename(cj)}')
    else:
        cam = json.load(open(cj))
        row['f_geocalib'] = cam['fx']; row['pitch_deg'] = cam.get('pitch_deg', np.nan)
        if (cam['width'], cam['height']) != (W, H):
            notes.append(f"GeoCalib size {cam['width']}x{cam['height']} != image {W}x{H} (different file or rotated?)")
        else:
            d, info = geometry_distance(pa, pb, cam, persons, e.get('cam_height_m'))
            if d is None: notes.append(info['note'])
            else:
                row.update(geometry_m=d, cam_height_m=info['h'], n_ref_people=info['n_ref'],
                           height_spread=np.nan if info['spread'] is None else info['spread'])

    # Depth Pro
    dz = f"{DEPTH_DIR}/{e.get('depth_stem', stem)}.npz"
    if not os.path.exists(dz):
        notes.append(f'no Depth Pro file {os.path.basename(dz)}')
    else:
        z = np.load(dz); depth = z['depth'].astype(np.float32); f_dp = float(z['f_px'])
        row['f_depthpro'] = f_dp
        if depth.shape != (H, W):
            notes.append(f'Depth Pro map {depth.shape[1]}x{depth.shape[0]} != image {W}x{H} (different file or rotated?)')
        else:
            row['depthpro_m'] = depthpro_distance(pa, pb, depth, f_dp)
    row['notes'] = '; '.join(notes)
    rows.append(row)

df = pd.DataFrame(rows)
df['ensemble_m']  = df[['geometry_m', 'depthpro_m']].mean(axis=1, skipna=False)      # plain mean of the two
df['agree_pct']   = 100 * (df.geometry_m - df.depthpro_m).abs() / df[['geometry_m', 'depthpro_m']].mean(axis=1, skipna=False)
df['methods_agree'] = df.agree_pct <= AGREE_TOL
df['f_ratio_dp_over_geo'] = df.f_depthpro / df.f_geocalib
for m in ('geometry', 'depthpro', 'ensemble'):
    df[f'{m}_err_m']   = df[f'{m}_m'] - df.true_m                          # signed: + means over-estimate
    df[f'{m}_err_pct'] = 100 * df[f'{m}_err_m'] / df.true_m
df.round(2).to_csv(f'{RES_DIR}/distance_accuracy_per_image.csv', index=False)
show_cols = ['image', 'true_m', 'geometry_m', 'depthpro_m', 'ensemble_m', 'agree_pct', 'cam_height_m', 'f_geocalib', 'f_depthpro', 'notes']
df[show_cols].round(2)

## Step 3 - Accuracy summary
`err` is signed (+ = too far); MAE / RMSE use absolute / squared error. **Paired** = only the photos where *both* methods produced a number, so the comparison is fair.

In [ ]:
def summarize(d):
    out = {}
    for m in ('geometry', 'depthpro', 'ensemble'):
        e, ep = d[f'{m}_err_m'], d[f'{m}_err_pct']
        out[m] = dict(n=int(e.notna().sum()), MAE_m=e.abs().mean(), median_abs_err_m=e.abs().median(),
                      RMSE_m=float(np.sqrt((e ** 2).mean())) if e.notna().any() else np.nan,
                      MAPE_pct=ep.abs().mean(), bias_pct=ep.mean())
    return pd.DataFrame(out).T.round(2)

paired = df.dropna(subset=['geometry_m', 'depthpro_m'])
print(f'photos: {len(df)} | geometry valid: {df.geometry_m.notna().sum()} | Depth Pro valid: {df.depthpro_m.notna().sum()} | both: {len(paired)}\n')
print('ALL valid photos per method (n differs between methods):'); display(summarize(df))
print('PAIRED photos only (fair comparison - use this one in the report):'); display(summarize(paired))
summarize(df).to_csv(f'{RES_DIR}/summary_all.csv'); summarize(paired).to_csv(f'{RES_DIR}/summary_paired.csv')

print('Does agreement between the two methods signal a reliable estimate? (paired photos; small n - indicative only)')
if len(paired):
    g = paired.assign(group=np.where(paired.methods_agree, f'agree (<= {AGREE_TOL:.0f}%)', f'disagree (> {AGREE_TOL:.0f}%)'))
    display(g.groupby('group').agg(photos=('image', 'count'), geometry_abs_err_m=('geometry_err_m', lambda s: s.abs().mean()),
                                   depthpro_abs_err_m=('depthpro_err_m', lambda s: s.abs().mean()),
                                   ensemble_abs_err_m=('ensemble_err_m', lambda s: s.abs().mean())).round(2))

print('Focal length check: Depth Pro estimates its own focal length; a ratio far from 1 means the two methods see the camera differently.')
display(df[['image', 'f_geocalib', 'f_depthpro', 'f_ratio_dp_over_geo']].round(2))

## Step 4 - Plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
lim = np.nanmax(df[['true_m', 'geometry_m', 'depthpro_m']].values) * 1.1
ax[0].plot([0, lim], [0, lim], 'k--', lw=1, label='perfect')
for m, c, mk in (('geometry', 'tab:blue', 'o'), ('depthpro', 'tab:orange', 's'), ('ensemble', 'tab:green', '^')):
    ax[0].scatter(df.true_m, df[f'{m}_m'], c=c, marker=mk, label=m, alpha=0.85)
ax[0].set_xlabel('tape-measured distance (m)'); ax[0].set_ylabel('estimated distance (m)'); ax[0].set_xlim(0, lim); ax[0].set_ylim(0, lim)
ax[0].set_title('Estimated vs true floor distance'); ax[0].legend(); ax[0].grid(alpha=0.3)

x = np.arange(len(df)); w = 0.27
for k, (m, c) in enumerate((('geometry', 'tab:blue'), ('depthpro', 'tab:orange'), ('ensemble', 'tab:green'))):
    ax[1].bar(x + (k - 1) * w, df[f'{m}_err_pct'], w, color=c, label=m)
ax[1].axhline(0, color='k', lw=0.8); ax[1].set_xticks(x); ax[1].set_xticklabels(df.image, rotation=45, ha='right')
ax[1].set_ylabel('signed error (% of true distance)'); ax[1].set_title('Error per photo (+ = estimated too far)'); ax[1].legend(); ax[1].grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(f'{RES_DIR}/distance_accuracy.png', dpi=130); plt.show()
print('saved to', RES_DIR)

## How to read this (and what to say)
* **Headline numbers:** the *paired* table - mean absolute error in metres and in % of the true distance, for each method and for their mean.
* **Bias:** a consistent + or - sign in `bias_pct` means a systematic scale error (for geometry, usually the 1.7 m assumption or the floor not being flat; for Depth Pro, its learned scale prior).
* **Agreement table:** if the photos where the two methods agree have clearly smaller errors, agreement is a usable label-free confidence signal for the demo.
* **Focal length ratio:** if Depth Pro's focal length is far from GeoCalib's on a photo, expect the two distances to disagree on it, and treat that photo's numbers with care.
* **Limits to state:** a handful of photos, tape error of a few cm, and the 1.7 m height assumption for geometry. This shows the size of the error on these photos; it is not a guarantee for every site.